# reachscan — run it yourself on a real model

You are about to be your **own first user**: install the instrument from the repo, point it at a model you didn't train it on, and watch target reachability move as a committed prefix deepens. If the gap appears here, the result isn't a Qwen-floor-sum artifact.

**Before you start**
- Runtime → Change runtime type → **GPU** (an L4/A100 is comfortable for 8B; T4 works for the smoke test).
- Llama-3.1 is **gated**: accept the license on its Hugging Face page first, then paste a token below. Don't have access yet? Use the ungated smoke model in the next cell to verify the pipeline in ~2 minutes.


## 1 · Install reachscan from your repo

In [ ]:
# Point this at your repo once reachscan is pushed. Until then, change to your branch/path.
REPO_URL = "git+https://github.com/ReachabilityLabs/ReachScan.git"   # <-- adjust to your real repo/branch

import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", f"reachscan[hf,plot] @ {REPO_URL}"])

# Fallback if not yet public — upload the zip to Colab and:
# Offline fallback: upload reachscan_v0_2_PUBLIC.zip to /content, then:
# !cd /content && unzip -q reachscan_v0_2_PUBLIC.zip && pip install -q "./reachscan[hf,plot]"
import reachscan; print('reachscan', reachscan.__version__, 'installed')

## 2 · Pick a model
Start with the **smoke model** to confirm the repackaged `hf_source.py` actually runs on weights (this is the one thing the test suite can't check). Then switch to Llama for the real run.

In [ ]:
# SMOKE (ungated, fast — verifies the file works on a real model):
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

# REAL RUN (gated — accept the license on HF, then run the login cell below):
# MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"

# Only needed for gated models like Llama:
# from huggingface_hub import login; login()  # paste your HF token

from reachscan.hf_source import HuggingFaceSource
source = HuggingFaceSource(MODEL_ID)
print('loaded', source.name)

## 3 · Define the problem and the projection
`ExactMatch(ground_truth)` measures reachability of the **exact** correct answer — the cleanest target-relative signal. Swap in `ModuloProjection(8, target_residue=...)` to see family-level foreclosure instead. Set the problem and its known correct answer.

In [ ]:
from reachscan import ExactMatch, ModuloProjection, SamplerPolicy, GeneratedPrefixSource

PROMPT = (
    "Please reason step by step, and put your final answer within \\boxed{}. "
    "Problem: Compute the sum of floor((3n+7)/5) for n = 1 through n = 40."
)
GROUND_TRUTH = 532          # <-- the known correct answer for THIS problem

projection = ExactMatch(GROUND_TRUTH)
# projection = ModuloProjection(8, target_residue=GROUND_TRUTH % 8)  # family view
print('projection:', projection.name)

## 4 · Smoke run — does the real-model path work end to end?
Tiny budget, two depths. If this prints a table with sane numbers, the repackaged HF source is verified on weights. Only then spend GPU-hours on the full scan.

In [ ]:
from reachscan import reach_scan, DepthSpec, uniform_plan

smoke_prefix = GeneratedPrefixSource(
    source, PROMPT, trace_sampler=SamplerPolicy(max_new_tokens=512), seed=0
)
smoke = reach_scan(
    source=source, prefix_source=smoke_prefix, projection=projection,
    plan=uniform_plan([0.0, 1.0], 8),
    rollout_sampler=SamplerPolicy(max_new_tokens=512), base_seed=0,
)
for s in smoke.summaries:
    print(f'f={s.fraction:.2f}  numeric={s.numeric:>2}  R_T={s.target_reachability:.3f}  dom={s.dominant_bucket}')
print('\nSmoke OK — the file runs on weights.' )

## 5 · Full reach-scan
Generate one reference trace, then scan committed prefixes of it. Per-depth `M` follows the paper's pattern (more rollouts where it matters). This is the run that takes real GPU time.

In [ ]:
plan = [
    DepthSpec(0.00, 128),
    DepthSpec(0.25,  64),
    DepthSpec(0.50,  64),
    DepthSpec(0.75, 128),
    DepthSpec(0.90, 128),
    DepthSpec(1.00, 128),
]

prefix_source = GeneratedPrefixSource(
    source, PROMPT, trace_sampler=SamplerPolicy(max_new_tokens=2048), seed=0
)
result = reach_scan(
    source=source, prefix_source=prefix_source, projection=projection,
    plan=plan, rollout_sampler=SamplerPolicy(max_new_tokens=512), base_seed=0,
)

print(f"{'depth':>6} {'M':>5} {'numeric':>7} {'R_T':>6} {'dom':>6} {'dom_mass':>8} {'entropy':>7}")
for s in result.summaries:
    print(f'{s.fraction:>6.2f} {s.attempts:>5} {s.numeric:>7} {s.target_reachability:>6.3f} '
          f'{str(s.dominant_bucket):>6} {s.dominant_mass:>8.3f} {s.answer_field_entropy:>7.3f}')

## 6 · Save the artifacts (this run becomes your real example output)

In [ ]:
from reachscan.metadata import write_result
from pathlib import Path
out = write_result(result, Path('reachscan_llama_run'))
print('artifacts written to', out)
!ls -la reachscan_llama_run

## 7 · Plot the gap

In [ ]:
import matplotlib.pyplot as plt
f   = [s.fraction for s in result.summaries]
rt  = [s.target_reachability for s in result.summaries]
ent = [s.answer_field_entropy for s in result.summaries]
fig, ax = plt.subplots(figsize=(7,4))
ax.plot(f, rt, 'o-', color='#B87333', label='target reachability R_T(f)')
ax.set_xlabel('committed-prefix depth'); ax.set_ylabel('R_T(f)'); ax.set_ylim(0,1)
ax2 = ax.twinx(); ax2.plot(f, ent, 's--', color='#0D9B8A', alpha=.6, label='answer-field entropy')
ax2.set_ylabel('entropy (bits)')
ax.set_title(f'reach-scan: {source.name}'); ax.legend(loc='upper left'); ax2.legend(loc='upper right')
plt.tight_layout(); plt.show()

## 8 · How to read it
- **A sustained fall in R_T(f) before answer exposure** is evidence of foreclosure under the declared sampler — the committed prefix is losing access to the correct answer. A single noisy run can wiggle, so look for a sustained trend, not one dip.
- **Late depths, especially f=1.0**, may already contain the answer in the committed trace; treat those as post-hoc morphology unless you check answer exposure.
- **Entropy dropping while R_T falls** is the field concentrating into a wrong basin (capture, not diffusion).
- **A `dom` bucket that isn't the target taking over** is where the wrong family wins.

Honesty rider: this is one model on one problem under one sampler. It is a *measurement*, not a law. What it earns you is a second substrate — and if the shape appears on a model you didn't tune the instrument on, that is the strongest evidence the effect isn't a Qwen artifact. Those saved artifacts are your first real, GPU-free-inspectable example output.